# Kafka inference demonstration: historical India AQI

**Samuel Flores and Santiago Becerra · ETL G51**

This notebook coordinates the real Python producer and consumer. It uses the validated PostgreSQL warehouse and the serialized AQI estimation model. It creates no synthetic observations.

Run it in **local Jupyter** from this repository using the same Python environment as the application. The setup in the README must already have started PostgreSQL and Kafka, populated the approved warehouse, and trained `artifacts/model.joblib` with its matching `metadata.json`.

This is not a Colab-ready Docker notebook: an ordinary Colab runtime does not provide this local Docker environment. The earlier Colab EDA/ETL notebooks remain preserved separately.


## 1. Verify paths and configuration

Set `INDIA_DATABASE_URL` in the environment of the Jupyter kernel to the host PostgreSQL connection. For the Compose stack the database port is 55432; Kafka is exposed at localhost:19092. Credentials belong in the local environment, not in notebook output or committed cells.

The consumer and producer run as separate processes. A fresh demo topic and prediction store isolate this notebook from earlier messages. The normal application topic remains `india.aqi.features.v1`.


In [ ]:
from pathlib import Path
import json
import os
import sqlite3
import socket
import subprocess
import sys
import time
import uuid

ROOT = Path.cwd().resolve()
if not (ROOT / "src" / "final_stream.py").exists():
    ROOT = ROOT.parent
assert (ROOT / "src" / "final_stream.py").exists(), "Open this notebook from the repository or its notebooks directory."
assert (ROOT / "artifacts" / "model.joblib").exists(), "Train the model first; see the README."
assert (ROOT / "artifacts" / "metadata.json").exists(), "Matching model metadata is required."
assert os.environ.get("INDIA_DATABASE_URL"), "Set INDIA_DATABASE_URL in the kernel environment first."

# Check the dashboard port before starting any notebook-owned process.
DEMO_DASHBOARD_PORT = int(os.environ.get("DEMO_DASHBOARD_PORT", "8503"))
if not 1 <= DEMO_DASHBOARD_PORT <= 65535:
    raise ValueError("DEMO_DASHBOARD_PORT must be between 1 and 65535.")
with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as probe:
    try:
        probe.bind(("127.0.0.1", DEMO_DASHBOARD_PORT))
    except OSError:
        raise RuntimeError(f"Dashboard port {DEMO_DASHBOARD_PORT} is unavailable. Choose a free DEMO_DASHBOARD_PORT before starting this notebook.") from None

run_id = uuid.uuid4().hex
TOPIC = "india.aqi.features.demo." + run_id[:12]
GROUP = "india-notebook-" + run_id[:12]
BOOTSTRAP = os.environ.get("KAFKA_BOOTSTRAP_SERVERS", "localhost:19092")
STORE = ROOT / "runtime" / ("demo-" + run_id[:12] + ".sqlite")
LOG_DIR = ROOT / "runtime" / ("notebook-" + run_id[:12])
LOG_DIR.mkdir(parents=True, exist_ok=True)
ENV = os.environ.copy()
ENV["PYTHONPATH"] = str(ROOT / "src")
ENV["INDIA_PROJECT_ROOT"] = str(ROOT)
ENV["PREDICTION_STORE"] = str(STORE)
ENV["MODEL_ARTIFACTS"] = str(ROOT / "artifacts")
print("Repository:", ROOT.name)
print("Demo topic:", TOPIC)
print("Prediction store:", STORE.name)


## 2. Inspect the trained model and reserved test evaluation

Each message carries one city's pollutant readings for one day, and the target is the AQI of that same day. The model was selected using 2022 validation data and fitted through 2022; the standard replay below requests 2023 records, which the model never saw. Error metrics and baseline comparisons describe offline estimation quality. They do not become better because messages travel through Kafka.


In [ ]:
metadata = json.loads((ROOT / "artifacts" / "metadata.json").read_text(encoding="utf-8"))
print("Model:", metadata["model_name"])
print("Version:", metadata["model_version"])
print("Fitted through date:", metadata["fit_through"])
print("Features:", metadata["feature_columns"])
print(json.dumps(metadata["metrics"]["test"], indent=2))
print("Beats mean baseline:", metadata["beats_mean_baseline_on_test"])


## 3. Create the real Kafka topic and start the consumer

The topic has one partition. The consumer loads the fitted pipeline once, validates each JSON event and writes a prediction or rejection before confirming its offset. It never reads the dimensional warehouse.

This bounded demo uses 50 messages. Start the dashboard in the next section before starting the producer to observe results arriving.


In [ ]:
script = str(ROOT / "src" / "final_stream.py")
subprocess.run([sys.executable, script, "create-topic", "--bootstrap", BOOTSTRAP,
                "--topic", TOPIC], cwd=ROOT, env=ENV, check=True, timeout=60)

processes = {}
log_handles = []

def launch(name, arguments):
    log_path = LOG_DIR / (name + ".log")
    handle = log_path.open("w", encoding="utf-8")
    log_handles.append(handle)
    flags = subprocess.CREATE_NO_WINDOW if os.name == "nt" else 0
    process = subprocess.Popen(arguments, cwd=ROOT, env=ENV,
                               stdout=handle, stderr=subprocess.STDOUT,
                               creationflags=flags)
    processes[name] = process
    return process

consumer = launch("consumer", [sys.executable, script, "consume",
    "--bootstrap", BOOTSTRAP, "--topic", TOPIC, "--group", GROUP,
    "--artifacts", str(ROOT / "artifacts"), "--store", str(STORE),
    "--max-messages", "50", "--idle-timeout", "180"])
print("Consumer process started. Check its log if it exits before publication.")


## 4. Start the Streamlit dashboard

The dashboard reads this consumer's output store, not PostgreSQL. The fragment refreshes every two seconds. Set `DEMO_DASHBOARD_PORT` in the kernel environment to choose a free local port; the default is 8503, separate from the standard Compose dashboard at 8502. The cell checks that the port is available and waits for Streamlit's health endpoint before showing a link. Open the link, leave the page visible and run the producer cell below.


In [ ]:
from urllib.request import urlopen
from urllib.error import URLError

def stop_own_demo_processes():
    for process in processes.values():
        if process.poll() is None:
            process.terminate()
            try:
                process.wait(timeout=5)
            except subprocess.TimeoutExpired:
                process.kill()
                process.wait(timeout=5)

dashboard = launch("dashboard", [sys.executable, "-m", "streamlit", "run",
    str(ROOT / "app" / "dashboard.py"), "--server.address=127.0.0.1",
    f"--server.port={DEMO_DASHBOARD_PORT}", "--server.headless=true", "--browser.gatherUsageStats=false"])
dashboard_url = f"http://127.0.0.1:{DEMO_DASHBOARD_PORT}"
deadline = time.monotonic() + 20
while True:
    if dashboard.poll() is not None:
        stop_own_demo_processes()
        raise RuntimeError(f"Streamlit exited before becoming ready. Inspect the local log in {LOG_DIR.name}/dashboard.log.")
    try:
        with urlopen(dashboard_url + "/_stcore/health", timeout=1) as response:
            healthy = response.status == 200 and response.read().strip() == b"ok"
        if healthy and dashboard.poll() is None:
            break
    except (URLError, TimeoutError, OSError):
        pass
    if time.monotonic() >= deadline:
        stop_own_demo_processes()
        raise TimeoutError(f"Streamlit was not healthy within 20 seconds. Inspect the local log in {LOG_DIR.name}/dashboard.log.")
    time.sleep(0.25)
from IPython.display import display, Markdown
display(Markdown(f"[Open the demo dashboard]({dashboard_url})"))


## 5. Publish 50 approved historical pollutant readings

The producer queries the validated fact-table view, builds the same features as training and publishes one event every 0.5 seconds. Publication time is current; the measurement date is historical. The observed AQI of the day and its category are not included in the message.


In [ ]:
producer = launch("producer", [sys.executable, script, "produce",
    "--bootstrap", BOOTSTRAP, "--topic", TOPIC,
    "--start", "2023-01-01", "--end", "2023-12-31",
    "--limit", "50", "--delay", "0.5", "--replay-id", run_id])
print("Replay is running in a separate process. Watch the dashboard counter and chart.")


## 6. Verify durable Kafka predictions

Run this after the replay. A successful process exit and 50 stored rows provide evidence for this bounded run. Kafka topic, partition and offset identify actual consumed records. This does not replace the quality-gate and leakage tests in the test suite.


In [ ]:
assert producer.wait(timeout=60) == 0, (LOG_DIR / "producer.log").read_text(encoding="utf-8")
assert consumer.wait(timeout=60) == 0, (LOG_DIR / "consumer.log").read_text(encoding="utf-8")
assert STORE.exists(), "The consumer did not create its output store."
import pandas as pd
with sqlite3.connect(STORE) as connection:
    count = connection.execute("SELECT COUNT(*) FROM predictions WHERE replay_id=?", (run_id,)).fetchone()[0]
    rejected = connection.execute("SELECT COUNT(*) FROM rejected_events").fetchone()[0]
    results = pd.read_sql_query(
        "SELECT city_name, origin_date, target_date, predicted_aqi, model_version, "
        "topic, partition_id, kafka_offset, processed_at FROM predictions "
        "WHERE replay_id=? ORDER BY kafka_offset", connection, params=[run_id])
assert count == 50, f"Expected 50 predictions; received {count}. Inspect consumer.log."
assert rejected == 0, f"Unexpected rejections: {rejected}"
print("Confirmed:", count, "predictions from Kafka; rejected:", rejected)
display(results.head(10))
print((LOG_DIR / "producer.log").read_text(encoding="utf-8"))
print((LOG_DIR / "consumer.log").read_text(encoding="utf-8").splitlines()[-1])


## 7. Demonstrate idempotent redelivery

A fresh consumer group reads the same 50 retained Kafka messages into the same output store. The stored event IDs already exist, so the consumer should report duplicates while the prediction count stays 50. Messages are retained in Kafka after consumption.


In [ ]:
duplicate_group = GROUP + "-redelivery"
result = subprocess.run([sys.executable, script, "consume",
    "--bootstrap", BOOTSTRAP, "--topic", TOPIC, "--group", duplicate_group,
    "--artifacts", str(ROOT / "artifacts"), "--store", str(STORE),
    "--max-messages", "50", "--idle-timeout", "30"],
    cwd=ROOT, env=ENV, check=True, capture_output=True, text=True, timeout=60)
summary = json.loads(result.stdout.strip().splitlines()[-1])
with sqlite3.connect(STORE) as connection:
    after = connection.execute("SELECT COUNT(*) FROM predictions WHERE replay_id=?", (run_id,)).fetchone()[0]
assert summary["duplicate"] == 50 and summary["predicted"] == 0, summary
assert after == 50
print(summary)
print("Prediction count remains:", after)


## 8. Stop only the processes started by this notebook

This stops the notebook's dashboard and any still-running demo processes. Logs and prediction results remain available. It does not delete broker topics, database records or Docker volumes, and it does not stop other application services.


In [ ]:
for name, process in processes.items():
    if process.poll() is None:
        process.terminate()
        try:
            process.wait(timeout=10)
        except subprocess.TimeoutExpired:
            print(name, "is still shutting down; stop it manually if needed.")
for handle in log_handles:
    handle.close()
print("Demo processes stopped. Logs:", LOG_DIR.name)
